# 03 — QLoRA Fine-Tuning

Fine-tune `Qwen/Qwen3-4B-Thinking-2507` with QLoRA (NF4 + LoRA rank-64) on the
public competition set plus a 20K subset of NuminaMath-CoT.

**Rejection-sampling SFT:** if a prior inference run exists
(`adaptive_public_v2_results.jsonl`), the model's own correct completions are used
as training targets. Incorrect or missing completions fall back to the gold answer.
NuminaMath-CoT examples provide additional rich CoT signal across a wider problem
distribution.

**Run order:** 02 (full public inference) → this notebook → 04 (GRPO) → 05 (private).


In [ ]:
# ── Google Colab Setup ──────────────────────────────────────────────────────
# No-op when running locally.  On Colab: installs packages, mounts Drive.
import sys

try:
    import google.colab  # noqa: F401
    _IS_COLAB = True
except ImportError:
    _IS_COLAB = False

if _IS_COLAB:
    print("Colab detected — installing packages (takes ~3-5 min)...")
    import subprocess
    pkgs = [
        "vllm>=0.6.0", "transformers>=4.45.0", "bitsandbytes>=0.43.0",
        "peft>=0.12.0", "trl>=0.12.0", "datasets", "accelerate>=0.34.0",
        "sympy>=1.12", "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
    print("Packages installed. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted. Upload public.jsonl / private.jsonl to")
    print("  MyDrive/math-qa-llm/data/raw/  before running further cells.")
else:
    print("Running locally — skipping Colab setup.")


In [ ]:
import json, os, re, sys
from pathlib import Path


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"

LORA_RANK    = 64
LORA_ALPHA   = 128
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

LEARNING_RATE  = 5e-5
BATCH_SIZE     = 1
GRAD_ACCUM     = 8
EPOCHS         = 2
MAX_SEQ_LENGTH = 4096     # keep at 4096 — safe on A30 24GB NF4 + grad_accum=8
WARMUP_STEPS   = 50
NUMINA_SUBSET  = 15_000   # was 5_000 — A30 24GB confirmed; more data, similar time budget

PUBLIC_PATH  = REPO_ROOT / "data" / "raw" / "public.jsonl"
PREV_RESULTS = REPO_ROOT / "artifacts" / "logs" / "runs" / "adaptive_public_v2_results.jsonl"
ADAPTER_OUT  = REPO_ROOT / "artifacts" / "models" / "qlora_v1"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    DRIVE_BASE   = Path('/content/drive/MyDrive/math-qa-llm')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    PUBLIC_PATH  = DRIVE_BASE / "data" / "raw" / "public.jsonl"
    PREV_RESULTS = DRIVE_BASE / "artifacts" / "logs" / "runs" / "adaptive_public_v2_results.jsonl"
    ADAPTER_OUT  = DRIVE_BASE / "artifacts" / "models" / "qlora_v1"

print(f"IS_COLAB     : {IS_COLAB}")
print(f"REPO_ROOT    : {REPO_ROOT}")
print(f"public       : {PUBLIC_PATH.is_file()} | prev_results: {PREV_RESULTS.is_file()}")
print(f"NUMINA_SUBSET: {NUMINA_SUBSET:,}")
print(f"adapter_out  : {ADAPTER_OUT}")


In [ ]:
import subprocess, sys

for dep in ["peft", "trl>=0.12.0", "datasets", "accelerate"]:
    pkg = dep.split(">=")[0].split("==")[0]
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset, load_dataset

# DataCollatorForCompletionOnlyLM moved between TRL versions — import gracefully
try:
    from trl import DataCollatorForCompletionOnlyLM
except ImportError:
    try:
        from trl.trainer import DataCollatorForCompletionOnlyLM
    except ImportError:
        DataCollatorForCompletionOnlyLM = None
        print("DataCollatorForCompletionOnlyLM unavailable — will train on full sequence (minor accuracy impact)")

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 1. Build Training Data

In [ ]:
with open(PUBLIC_PATH, encoding="utf-8") as f:
    public_data = [json.loads(line) for line in f]

prev_results: dict[str, dict] = {}
if PREV_RESULTS.is_file():
    with open(PREV_RESULTS, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            prev_results[str(rec["id"])] = rec

def gold_to_boxed(item: dict) -> str:
    gold = item.get("answer")
    if isinstance(gold, list):
        return "\\boxed{" + ", ".join(str(g) for g in gold) + "}"
    return "\\boxed{" + str(gold) + "}"

train_items = []
for item in public_data:
    res = prev_results.get(str(item["id"]))
    if res and res.get("correct") and res.get("response"):
        train_items.append({"item": item, "target": res["response"], "src": "model"})
    elif item.get("answer") is not None:
        train_items.append({"item": item, "target": gold_to_boxed(item), "src": "gold"})

from_model = sum(1 for t in train_items if t["src"] == "model")
print(f"Public: {len(train_items)} examples ({from_model} model correct, {len(train_items)-from_model} gold)")

## 2. Load Tokenizer & Build Dataset

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics, "
    "from algebra and calculus to number theory and combinatorics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Identify what is given and what you need to find.\n"
    "2. PLAN: Write down the key equations, formulas, or theorems you will use.\n"
    "3. SOLVE: Work through each step carefully. Compute intermediate results explicitly. "
    "Pay special attention to arithmetic - do not skip steps.\n"
    "4. VERIFY: Check that your answer satisfies all conditions in the problem. "
    "Check units, sign, and order of magnitude.\n"
    "5. ANSWER: Put your final answer in \\boxed{}.\n\n"
    "Additional rules:\n"
    "- If the problem has multiple blanks ([ANS] placeholders), put ALL answers "
    "comma-separated in ONE \\boxed{} in the order they appear. "
    "Example: \\boxed{3, -7, 42}.\n"
    "- Simplify all fractions and radical expressions completely.\n"
    "- You\'d better be sure of your answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Read the problem and all answer choices carefully.\n"
    "2. PLAN: Identify the relevant concepts, formulas, or theorems that apply.\n"
    "3. SOLVE: Work through the problem step by step. Compute intermediate results "
    "explicitly - do not skip arithmetic steps.\n"
    "4. ELIMINATE: Cross out answer choices that are clearly wrong.\n"
    "5. VERIFY: Confirm your chosen answer is consistent with every condition in the problem.\n"
    "6. ANSWER: On the very last line of your response, write ONLY \\boxed{X} "
    "where X is the letter of the correct answer (A-J). "
    "Do not write any text after \\boxed{}.\n\n"
    "You\'d better be sure of your answer."
)


def build_prompt(question: str, options) -> tuple:
    if options:
        labels   = [chr(65 + i) for i in range(len(options))]
        opts_txt = "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_txt}"
    n_ans = question.count("[ANS]")
    if n_ans > 1:
        hint = (
            f"\n\n[Note: This problem has {n_ans} answers. "
            f"Put all {n_ans} answers comma-separated in ONE \\boxed{{}} "
            f"in the order they appear in the question.]"
        )
        return SYSTEM_PROMPT_MATH, question + hint
    return SYSTEM_PROMPT_MATH, question


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"


def format_public(t: dict) -> str:
    system, user = build_prompt(t["item"]["question"], t["item"].get("options"))
    msgs = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": t["target"]},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)


def format_numina(ex: dict) -> str:
    msgs = [
        {"role": "system",    "content": SYSTEM_PROMPT_MATH},
        {"role": "user",      "content": ex["problem"]},
        {"role": "assistant", "content": ex["solution"]},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)


public_texts = [format_public(t) for t in train_items]

print(f"Downloading NuminaMath-CoT {NUMINA_SUBSET:,} samples...")
nds = load_dataset("AI-MO/NuminaMath-CoT", split="train")
nds = nds.shuffle(seed=42).select(range(NUMINA_SUBSET))
numina_texts = [format_numina(ex) for ex in nds]

all_texts = public_texts + numina_texts
dataset   = Dataset.from_dict({"text": all_texts})
print(f"Dataset: {len(dataset)} examples ({len(public_texts)} public + {len(numina_texts)} NuminaMath)")

## 3. Load Model with QLoRA

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

alloc = torch.cuda.memory_allocated() / 1e9
print(f"VRAM allocated: {alloc:.2f} GB")

## 4. Train

In [ ]:
import inspect
import logging as _stdlogging
import warnings as _warnings
import transformers as _tf

# ── Silence all log lines from transformers/trl/peft ──────────────────────────
_tf.logging.set_verbosity_error()
for _name in ("transformers", "trl", "peft", "accelerate", "datasets"):
    _stdlogging.getLogger(_name).setLevel(_stdlogging.ERROR)
_warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"]            = "false"

RESPONSE_TEMPLATE = "<|im_start|>assistant\n"

collator     = None
use_collator = False
if DataCollatorForCompletionOnlyLM is not None:
    try:
        collator     = DataCollatorForCompletionOnlyLM(RESPONSE_TEMPLATE, tokenizer=tokenizer)
        use_collator = True
    except Exception:
        pass

# ── Build SFTConfig (TRL-version-agnostic) ────────────────────────────────────
_sft_cfg_params = set(inspect.signature(SFTConfig.__init__).parameters)

_sft_kwargs = dict(
    output_dir=str(ADAPTER_OUT / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    logging_steps=10_000_000,
    logging_strategy="no",
    log_level="error",
    save_steps=100,
    save_total_limit=2,
    save_strategy="steps",
    report_to="none",
    disable_tqdm=False,
    packing=False,
)
if "max_length" in _sft_cfg_params:
    _sft_kwargs["max_length"] = MAX_SEQ_LENGTH
elif "max_seq_length" in _sft_cfg_params:
    _sft_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
if "dataset_text_field" in _sft_cfg_params:
    _sft_kwargs["dataset_text_field"] = "text"
_sft_kwargs = {k: v for k, v in _sft_kwargs.items() if k in _sft_cfg_params}
sft_config  = SFTConfig(**_sft_kwargs)

# ── Build SFTTrainer (TRL-version-agnostic) ───────────────────────────────────
_sft_trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters)
_trainer_kwargs = dict(model=model, args=sft_config, train_dataset=dataset)
if use_collator:
    _trainer_kwargs["data_collator"] = collator
if "processing_class" in _sft_trainer_params:
    _trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in _sft_trainer_params:
    _trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**_trainer_kwargs)

# ── Rip out every "print each step" callback ──────────────────────────────────
# Order matters — Jupyter swaps ProgressCallback for NotebookProgressCallback,
# so we kill both then add back a plain tqdm-only ProgressCallback.
from transformers.trainer_callback import (
    PrinterCallback,
    ProgressCallback,
)
for _cb_cls in (PrinterCallback, ProgressCallback):
    try:
        trainer.remove_callback(_cb_cls)
    except Exception:
        pass
# Also remove NotebookProgressCallback if it's there (the HTML metrics table)
try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

# Re-add plain ProgressCallback (just the tqdm bar, no metrics table, no prints)
trainer.add_callback(ProgressCallback())

# Debug — print the final callback list so we can verify
print("Active callbacks:", [type(cb).__name__ for cb in trainer.callback_handler.callbacks])

# ── Auto-resume from latest checkpoint if pod died mid-training ───────────────
_ckpt_root   = Path(sft_config.output_dir)
_latest_ckpt = None
if _ckpt_root.is_dir():
    _ckpts = sorted(_ckpt_root.glob("checkpoint-*"),
                    key=lambda p: int(p.name.split("-")[-1]))
    if _ckpts:
        _latest_ckpt = str(_ckpts[-1])
        print(f"Resuming from checkpoint: {_latest_ckpt}")
    else:
        print("No prior checkpoint — starting from step 0")
else:
    print("No prior checkpoint — starting from step 0")

print(f"Training {len(dataset)} examples, {EPOCHS} epochs, "
      f"effective batch {BATCH_SIZE * GRAD_ACCUM} — progress bar only:")
trainer.train(resume_from_checkpoint=_latest_ckpt)
print("Training complete.")

## 5. Save Adapter

In [ ]:
ADAPTER_OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_OUT))
tokenizer.save_pretrained(str(ADAPTER_OUT))
print(f"Adapter saved: {ADAPTER_OUT}")
print("Next: run 04_grpo_train.ipynb, or set RUN_MERGE=True below for vLLM inference.")

## 6. Merge Adapter (set RUN_MERGE=True when done with GRPO, or to skip GRPO)

In [ ]:
MERGE_OUTPUT = (DRIVE_BASE if IS_COLAB else REPO_ROOT) / "artifacts" / "models" / "qlora_v1_merged"
RUN_MERGE    = True

if RUN_MERGE:
    merged = model.merge_and_unload()
    MERGE_OUTPUT.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(str(MERGE_OUTPUT), safe_serialization=True)
    tokenizer.save_pretrained(str(MERGE_OUTPUT))
    print(f"Merged model saved: {MERGE_OUTPUT}")
    print(f"Set MODEL_ID = r'{MERGE_OUTPUT}' in 05_private_submission.ipynb")
else:
    print("Merge skipped (RUN_MERGE=False).")